In [31]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import scipy
from scipy.io import loadmat
import mne

In [32]:
import sys
import os

sys.path.append(r'C:\Users\torho\PycharmProjects\TranformerP300')

In [33]:
data = scipy.io.loadmat('data_kirasirova/S1901-P300_classic.mat')
X_real = data['epochs']
y_real = data['labels'].flatten()

In [34]:
from generate_data import build_sequences_new

In [35]:
ch_names = ['Fz', 'Cz', 'Pz']
sfreq = 250
info = mne.create_info(ch_names, sfreq, ch_types='eeg')
events = np.column_stack((np.arange(len(y_real)), np.zeros(len(y_real), int), y_real))
event_id = {'non-target': 0, 'target': 1}
epochs_mne = mne.EpochsArray(X_real, info, events=events, event_id=event_id)
epochs_mne.filter(l_freq=1, h_freq=15, filter_length='auto')

all_target = epochs_mne['target']
all_non_target = epochs_mne['non-target']

synth_sequences, synth_targets = build_sequences_new(
    all_target, all_non_target, n_classes=16, n_samples=2000
)


Not setting metadata
6759 matching events found
No baseline correction applied
0 projection items activated
Setting up band-pass filter from 1 - 15 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 15.00 Hz
- Upper transition bandwidth: 3.75 Hz (-6 dB cutoff frequency: 16.88 Hz)
- Filter length: 825 samples (3.300 s)



C:\Users\torho\AppData\Local\Temp\ipykernel_101608\563296646.py:7: RuntimeWarning: filter_length (825) is longer than the signal (250), distortion is likely. Reduce filter length or filter a longer signal.
  epochs_mne.filter(l_freq=1, h_freq=15, filter_length='auto')


In [36]:
synth_sequences.shape

(2000, 16, 3, 250)

In [37]:

batch_size = synth_sequences.shape[0]
X_collapsed = synth_sequences.reshape(batch_size, 1, 3, 16 * 250)  #в (2000, 1, 3, 4000)
X_collapsed.shape



(2000, 1, 3, 4000)

In [38]:
y_tensor = torch.LongTensor(synth_targets)

In [39]:
class SimpleTransformerCollapsed(nn.Module):
    def __init__(self, n_classes=16):
        super().__init__()
        
        #Пространственная свертка
        # (Batch, 1, 3, 4000) ->(Batch, 32, 1, 4000)
        self.spatial_conv = nn.Conv2d(1, 32, (3, 1), bias=False)
        self.bn1 = nn.BatchNorm2d(32)

        #Временная свертка (Batch, 32, 1, 4000) -> (Batch, 32, 1, 1000)
        self.temporal_conv = nn.Conv2d(32, 32, (1, 8), stride=(1, 4), padding=(0, 2))
        self.bn2 = nn.BatchNorm2d(32)

        #Пуллинг (Batch, 32, 1, 1000)->(Batch, 32, 1, 250)
        self.pool = nn.AdaptiveAvgPool2d((1, 250))
        
        self.pre_norm = nn.LayerNorm(32)
        self.pre_dropout = nn.Dropout(0.1)
        self.pre_linear = nn.Linear(32, 32)
        
        #pos. encoding
        self.pos_encoder = nn.Parameter(torch.randn(1, 250, 32) * 0.02)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=32, nhead=4, dim_feedforward=128,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.classifier = nn.Linear(32, n_classes)
        
    def forward(self, x):
        x = self.spatial_conv(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        
        x = self.temporal_conv(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        
        x = self.pool(x)                    
        x = x.squeeze(2)
        x = x.permute(0, 2, 1)
        
        x = self.pre_norm(x)
        x = self.pre_dropout(x)
        x = self.pre_linear(x)
        x = F.gelu(x)
        x = x + self.pos_encoder[:, :x.size(1), :]
        
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.classifier(x)

In [40]:
#санити чек
X_small = X_collapsed[:10]
y_small = y_tensor[:10]

X_small_tensor = torch.FloatTensor(X_small)
y_small_tensor = torch.LongTensor(y_small)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleTransformerCollapsed(n_classes=16).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

X_small_tensor = X_small_tensor.to(device)
y_small_tensor = y_small_tensor.to(device)

for epoch in range(100):
    optimizer.zero_grad()
    output = model(X_small_tensor)
    loss = criterion(output, y_small_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}, Loss: {loss.item():.6f}")

with torch.no_grad():
    output = model(X_small_tensor)
    pred = output.argmax(dim=1)
    acc = (pred == y_small_tensor).float().mean().item()
    print(f"точность {acc*100:.1f}%")
    print(f"предсказания {pred.cpu().tolist()}")
    print(f"истинные метки    {y_small_tensor.cpu().tolist()}")

Epoch  20, Loss: 1.959765
Epoch  40, Loss: 1.305839
Epoch  60, Loss: 0.658885
Epoch  80, Loss: 0.254971
Epoch 100, Loss: 0.136667
точность 100.0%
предсказания [6, 7, 14, 4, 7, 13, 8, 4, 5, 0]
истинные метки    [6, 7, 14, 4, 7, 13, 8, 4, 5, 0]


In [41]:
# трейн на усредненных: усреднение

from generate_data import build_averaged_dataset

In [42]:
#1. трейн на усредненных Κ=10

X_avg, y_avg = build_averaged_dataset(synth_sequences, synth_targets, K=10, repeats=50)


In [43]:
X_avg_collapsed = X_avg.reshape(X_avg.shape[0], 1, 3, 16 * 250)

X_avg_collapsed.shape


(800, 1, 3, 4000)

In [44]:
y_avg.shape

(800,)

In [45]:

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X_avg_collapsed, y_avg, test_size=0.2, random_state=42, stratify=y_avg
)

In [46]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.LongTensor(y_train).to(device)
X_val_t = torch.FloatTensor(X_val).to(device)
y_val_t = torch.LongTensor(y_val).to(device)
criterion = nn.CrossEntropyLoss()


In [47]:
def train_simple(X_train_t, y_train_t, X_val_t, y_val_t, n_epochs=40):
    model = SimpleTransformerCollapsed(n_classes=16).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)    
    
    best_val_acc = 0
    
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        output = model(X_train_t)
        loss = criterion(output, y_train_t)
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                train_acc = (model(X_train_t).argmax(dim=1) == y_train_t).float().mean().item()
                val_acc = (model(X_val_t).argmax(dim=1) == y_val_t).float().mean().item()
                
                print(f"  Epoch {epoch+1:3d}: Train Acc = {train_acc*100:.1f}%, Val Acc = {val_acc*100:.1f}%")
                
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
    
    return best_val_acc

In [48]:
def run_multiple_times_k10(X_train, y_train, X_val, y_val, n_runs=3):
    results = []
    
    for run in range(n_runs):
        print(f"run {run+1}")
        
        seed = 42 + run * 100
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        best_acc = train_simple(X_train_t, y_train_t, X_val_t, y_val_t, n_epochs=100)
        
        results.append(best_acc * 100)
        print(f"  Результат: {best_acc*100:.1f}%")
    
    print(f"для K=10")
    print(f"Средняя точность: {np.mean(results):.1f}% ± {np.std(results):.1f}%")
    print(f"Min: {np.min(results):.1f}%, Max: {np.max(results):.1f}%")
    
    return results

In [49]:
run_multiple_times_k10(X_train, y_train, X_val, y_val, n_runs=1)

run 1
  Epoch  10: Train Acc = 8.3%, Val Acc = 8.7%
  Epoch  20: Train Acc = 16.2%, Val Acc = 11.9%
  Epoch  30: Train Acc = 20.9%, Val Acc = 20.0%
  Epoch  40: Train Acc = 18.9%, Val Acc = 17.5%
  Epoch  50: Train Acc = 25.8%, Val Acc = 23.1%
  Epoch  60: Train Acc = 26.6%, Val Acc = 23.1%
  Epoch  70: Train Acc = 29.1%, Val Acc = 28.1%
  Epoch  80: Train Acc = 27.3%, Val Acc = 24.4%
  Epoch  90: Train Acc = 31.2%, Val Acc = 28.7%
  Epoch 100: Train Acc = 29.8%, Val Acc = 25.6%
  Результат: 28.7%
для K=10
Средняя точность: 28.7% ± 0.0%
Min: 28.7%, Max: 28.7%


[28.749999403953552]

In [50]:
run_multiple_times_k10(X_train, y_train, X_val, y_val, n_runs=3)

run 1
  Epoch  10: Train Acc = 8.3%, Val Acc = 8.7%
  Epoch  20: Train Acc = 16.2%, Val Acc = 11.9%
  Epoch  30: Train Acc = 20.9%, Val Acc = 20.0%
  Epoch  40: Train Acc = 18.9%, Val Acc = 17.5%
  Epoch  50: Train Acc = 25.8%, Val Acc = 23.1%
  Epoch  60: Train Acc = 26.6%, Val Acc = 23.1%
  Epoch  70: Train Acc = 29.1%, Val Acc = 28.1%
  Epoch  80: Train Acc = 27.3%, Val Acc = 24.4%
  Epoch  90: Train Acc = 31.2%, Val Acc = 28.7%
  Epoch 100: Train Acc = 29.8%, Val Acc = 25.6%
  Результат: 28.7%
run 2
  Epoch  10: Train Acc = 12.8%, Val Acc = 15.0%
  Epoch  20: Train Acc = 16.6%, Val Acc = 15.6%
  Epoch  30: Train Acc = 20.0%, Val Acc = 20.6%
  Epoch  40: Train Acc = 21.6%, Val Acc = 18.8%
  Epoch  50: Train Acc = 21.7%, Val Acc = 21.3%
  Epoch  60: Train Acc = 18.9%, Val Acc = 16.9%
  Epoch  70: Train Acc = 23.0%, Val Acc = 19.4%
  Epoch  80: Train Acc = 26.7%, Val Acc = 23.7%
  Epoch  90: Train Acc = 32.5%, Val Acc = 28.1%
  Epoch 100: Train Acc = 45.9%, Val Acc = 43.8%
  Результат

[28.749999403953552, 43.75, 33.75000059604645]